# ModelForge · Train specialists offline
Run this before the live demo. Select a GPU in **Runtime → Change runtime type**.
Both LoRA adapters are fine-tuned from the same pinned Qwen checkpoint.
This is not foundation-model pretraining. Starter examples demonstrate the pipeline;
they do not guarantee visible specialization. Colab Pro hardware and session time vary.
Save everything important to Drive and download a backup before judging.


## 1. Upload the project ZIP and install dependencies


In [ ]:
from google.colab import files
from pathlib import Path
import io, os, sys, zipfile, subprocess

uploaded = files.upload()  # Upload modelforge.zip from this delivery.
if len(uploaded) != 1:
    raise ValueError("Upload exactly one project ZIP.")
project_root = Path("/content")
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for member in archive.infolist():
        destination = (project_root / member.filename).resolve()
        if not destination.is_relative_to(project_root):
            raise ValueError("Unsafe archive path")
    archive.extractall(project_root)
PROJECT = Path("/content/modelforge")
if not (PROJECT / "pyproject.toml").exists():
    raise ValueError("ZIP must contain modelforge/pyproject.toml")
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,dev]", "ipywidgets"], check=True)
sys.path.insert(0, str(PROJECT / "backend"))
import torch
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun the notebook.")


## 2. Mount Drive and choose a training budget
Start with two steps to check the pipeline; then increase STEPS and rerun in a fresh output directory.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/ModelForge/adapters")
OUTPUT.mkdir(parents=True, exist_ok=True)
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
STEPS = 100
RANK = 8
MAX_LENGTH = 384
subprocess.run([sys.executable, "training/prepare_data.py", "--output", "data"], check=True)
print("Adapters and optimizer checkpoints will persist at", OUTPUT)


## 3. Train coding, then writing
Each starts from the same foundation, not from the other specialist. Measured training time and validation loss are recorded. Run cells separately to inspect progress.


In [ ]:
import time
started = time.perf_counter()
subprocess.run([sys.executable, "training/train.py", "--skill", "coding", "--base-model", BASE_MODEL,
                "--output", str(OUTPUT), "--steps", str(STEPS), "--rank", str(RANK),
                "--max-length", str(MAX_LENGTH)], check=True)
print("Coding elapsed minutes:", round((time.perf_counter() - started) / 60, 2))


In [ ]:
started = time.perf_counter()
subprocess.run([sys.executable, "training/train.py", "--skill", "writing", "--base-model", BASE_MODEL,
                "--output", str(OUTPUT), "--steps", str(STEPS), "--rank", str(RANK),
                "--max-length", str(MAX_LENGTH)], check=True)
print("Writing elapsed minutes:", round((time.perf_counter() - started) / 60, 2))


## Optional: train Humor and Thinking
Use the same pinned foundation. Starter examples demonstrate the pipeline and do not establish skill quality.


In [ ]:
for skill in ("humor", "thinking"):
    subprocess.run([sys.executable, "training/train.py", "--skill", skill, "--base-model", BASE_MODEL,
                    "--output", str(OUTPUT), "--steps", str(STEPS), "--rank", str(RANK),
                    "--max-length", str(MAX_LENGTH)], check=True)


## 4. Validate compatibility and download a backup


In [ ]:
import json, shutil
from modelforge.merge import validate_pair
loaded = validate_pair({skill: OUTPUT / skill for skill in ("coding", "humor", "thinking", "writing") if (OUTPUT / skill / "adapter_model.safetensors").exists()}, BASE_MODEL, None)
print(json.dumps({k: {"rank": v[0]["r"], "revision": v[0]["revision"], "sha256": v[2]} for k,v in loaded.items()}, indent=2))
archive_path = shutil.make_archive("/content/modelforge-trained-adapters", "zip", OUTPUT)
files.download(archive_path)


## Resume after interruption
Find the newest `checkpoint-N` under the skill's `checkpoints` directory, then use
`python training/train.py --skill coding --output /content/drive/MyDrive/ModelForge/adapters --steps 200 --resume PATH_TO_CHECKPOINT`.
Keep the same model, rank, sequence length and training data when resuming optimizer state.
Next open **02_live_lab.ipynb**. Do not claim skill improvement from loss alone; compare on held-out tasks.
